# BCWD Case Interpretation Across Models

- 대상 데이터셋: `Breast_Cancer_Wisconsin_(Original)`
- 비교 모델: `ANFIS`, `GA-ANFIS`, `PSO-ANFIS`, `H-ANFIS`, `GRS-ANFIS`
- 이 노트북은 **실제 malignant(유방암) 환자 1명**을 골라서 같은 케이스를 각 모델이 어떻게 해석하는지 보여줍니다.
- E404 전처리 기준에서 BCWD는 `9`개 raw numeric feature를 그대로 사용합니다.
- 따라서 아래의 환자 원시값과 모델 입력값은 동일한 `9`개 변수이며, 체크포인트에 저장된 feature 순서와 scaler만 맞춰 각 모델 규칙을 해석합니다.
- `GRS-ANFIS`는 malignant validation 케이스 중에서 `Primary(base)`는 malignant 확률이 높게 보지만 `Complementary(residual)`를 더한 뒤 `full` 예측이 가장 많이 달라지는 샘플을 우선 선택합니다.
- 기본 설정에서는 각 rule의 antecedent를 최대 `9`개 조건까지 보여주므로, BCWD에서는 사실상 전체 입력 조건을 확인할 수 있습니다.

설정은 아래 셀의 `FOLD_IDX`, `CASE_INDEX_OVERRIDE`, `H_MODEL_NAME`만 바꾸면 됩니다.



In [1]:
from pathlib import Path
import json
import os
import sys
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd().resolve()


def _looks_like_project_root(path: Path) -> bool:
    markers = ['model.py', 'data.py', 'learning.py', 'gh_config.py']
    return all((path / marker).exists() for marker in markers)


if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_E404',
        PROJECT_ROOT / 'GH-ANFIS_E404',
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_E403',
        PROJECT_ROOT / 'GH-ANFIS_E403',
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import GH_ANFIS, TSKANFIS, ParallelHierarchicalTSKANFIS
from gh_config import normalize_gh_params
from data import load_bcwd_data, coerce_numeric_frame, drop_nan_targets

PROJECT_ROOT


PosixPath('/home/harp3133t/Research/03_Research/GH-ANFIS_E404')

In [2]:
def safe_name(name: str) -> str:
    return ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(name))


SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATASET_NAME = 'Breast_Cancer_Wisconsin_(Original)'
RAW_BCWD_CSV = PROJECT_ROOT / 'data' / 'bcwd_uci_15.csv'
CV_WEIGHT_ROOT = PROJECT_ROOT / 'hyper_parameter' / 'cv_weights'
EXPORT_DIR = PROJECT_ROOT / 'output' / 'bcwd_case_model_interpretation'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

FOLD_IDX = 4
CASE_INDEX_OVERRIDE = None   # 예: 137 처럼 row index를 직접 지정하면 자동 탐색 대신 그 케이스를 사용
MIN_BASE_POSITIVE_PROB = 0.80
TOP_RULES_PER_MODEL = 3
TOP_TERMS_PER_RULE = 9
H_MODEL_NAME = 'PH-ANFIS(Avg)'   # 또는 'PH-ANFIS(Stacked)'
ARTIFACT_DATASET_DIRNAME = None  # 예: 'Breast_Cancer_Wisconsin__Original___no_mi'

MODEL_FILE_STEM = {
    'GH-ANFIS': 'gh_anfis',
    'ANFIS': 'anfis',
    'GA-ANFIS': 'ga_anfis',
    'PSO-ANFIS': 'pso_anfis',
    'PH-ANFIS(Avg)': 'ph_anfis_avg',
    'PH-ANFIS(Stacked)': 'ph_anfis_stacked',
}

MODEL_NAMES = ['ANFIS', 'GA-ANFIS', 'PSO-ANFIS', H_MODEL_NAME, 'GH-ANFIS']
MODEL_DISPLAY_NAMES = {
    'ANFIS': 'ANFIS',
    'GA-ANFIS': 'GA-ANFIS',
    'PSO-ANFIS': 'PSO-ANFIS',
    'PH-ANFIS(Avg)': 'H-ANFIS',
    'PH-ANFIS(Stacked)': 'H-ANFIS(Stacked)',
    'GH-ANFIS': 'GRS-ANFIS',
}

print('DEVICE =', DEVICE)
print('RAW_BCWD_CSV =', RAW_BCWD_CSV)
print('CV_WEIGHT_ROOT =', CV_WEIGHT_ROOT)
print('FOLD_IDX =', FOLD_IDX)
print('H_MODEL_NAME =', H_MODEL_NAME)


DEVICE = cuda
RAW_BCWD_CSV = /home/harp3133t/Research/03_Research/GH-ANFIS_E404/data/bcwd_uci_15.csv
CV_WEIGHT_ROOT = /home/harp3133t/Research/03_Research/GH-ANFIS_E404/hyper_parameter/cv_weights
FOLD_IDX = 4
H_MODEL_NAME = PH-ANFIS(Avg)


In [3]:
def sigmoid_np(x: np.ndarray | float) -> np.ndarray | float:
    x_clip = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x_clip))


def load_bcwd_views() -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, list[int], dict[int, int]]:
    raw_df = pd.read_csv(RAW_BCWD_CSV)
    if 'target' not in raw_df.columns:
        raise ValueError(f"Raw BCWD snapshot is missing 'target': {RAW_BCWD_CSV}")

    raw_x_df = raw_df.drop(columns=['target']).reset_index(drop=True)
    y_raw = pd.to_numeric(raw_df['target'], errors='raise').astype(int).to_numpy()

    processed_x_df, y_encoded, _ = load_bcwd_data()
    processed_x_df = coerce_numeric_frame(processed_x_df)
    processed_x_df, y_encoded = drop_nan_targets(processed_x_df, y_encoded)
    processed_x_df = processed_x_df.reset_index(drop=True)
    y_encoded = np.asarray(y_encoded, dtype=int)

    if len(raw_x_df) != len(processed_x_df):
        raise ValueError(
            f'Raw snapshot rows ({len(raw_x_df)}) and processed BCWD rows ({len(processed_x_df)}) do not match.'
        )

    raw_classes = sorted(pd.unique(y_raw).tolist())
    raw_to_encoded = {int(raw): idx for idx, raw in enumerate(raw_classes)}
    y_from_raw = np.asarray([raw_to_encoded[int(v)] for v in y_raw], dtype=int)
    if not np.array_equal(y_from_raw, y_encoded):
        raise ValueError('Raw target mapping and processed label encoding are not aligned.')

    return raw_x_df, processed_x_df, y_raw, y_encoded, raw_classes, raw_to_encoded


def get_bcwd_fold_indices(y_encoded: np.ndarray, fold_idx: int, seed: int = SEED, n_splits: int = 5):
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(splitter.split(np.zeros(len(y_encoded)), y_encoded))
    if int(fold_idx) < 1 or int(fold_idx) > len(splits):
        raise ValueError(f'fold_idx must be in [1, {len(splits)}], got {fold_idx}')
    train_idx, val_idx = splits[int(fold_idx) - 1]
    return np.asarray(train_idx), np.asarray(val_idx)


def resolve_artifact_dataset_dir(
    dataset_name: str,
    fold_idx: int,
    required_models: list[str],
    root_dir: Path = CV_WEIGHT_ROOT,
    preferred_dirname: str | None = None,
) -> Path:
    if preferred_dirname is not None:
        candidate = Path(root_dir) / str(preferred_dirname)
        if not candidate.exists():
            raise FileNotFoundError(f'Preferred artifact dataset dir not found: {candidate}')
        return candidate.resolve()

    dataset_key = safe_name(dataset_name)
    seen = set()
    candidates = []
    for cand in [Path(root_dir) / dataset_key, *sorted(Path(root_dir).glob(f'{dataset_key}*'))]:
        if not cand.exists() or not cand.is_dir():
            continue
        key = str(cand.resolve())
        if key in seen:
            continue
        seen.add(key)
        candidates.append(cand)

    if not candidates:
        raise FileNotFoundError(f'No artifact directories found for dataset key prefix: {dataset_key}')

    scored = []
    fold_name = f'fold_{int(fold_idx):02d}'
    for cand in candidates:
        fold_dir = cand / fold_name
        missing = []
        if not fold_dir.exists():
            missing.append('<missing fold dir>')
        else:
            for model_name in required_models:
                stem = MODEL_FILE_STEM[model_name]
                if not (fold_dir / f'{stem}.pt').exists():
                    missing.append(f'{stem}.pt')
        score = (
            0 if cand.name.endswith('___no_mi') else 1,
            len(missing),
            cand.name,
        )
        scored.append((score, cand, missing))

    valid = [(score, cand) for score, cand, missing in scored if not missing]
    if valid:
        valid.sort(key=lambda item: item[0])
        return valid[0][1].resolve()

    detail_lines = [f'{cand.name}: missing={missing}' for _, cand, missing in scored]
    raise FileNotFoundError(
        'Could not find an artifact directory containing all requested model checkpoints.\n'
        + '\n'.join(detail_lines)
    )



def _infer_model_config(payload: dict[str, Any], params: dict[str, Any]) -> dict[str, Any]:
    cfg = payload.get('model_config')
    if cfg is not None:
        return dict(cfg)

    n_features = payload.get('n_features')
    n_outputs = payload.get('n_outputs')
    if n_features is None or n_outputs is None:
        raise KeyError('Checkpoint must contain either model_config or n_features/n_outputs.')

    out = {
        'n_features': int(n_features),
        'n_outputs': int(n_outputs),
    }
    if 'base_rules' in params:
        out['base_rules'] = int(params['base_rules'])
    if 'residual_rules' in params:
        out['residual_rules'] = int(params['residual_rules'])
    if 'mf_per_feature' in params:
        out['mf_per_feature'] = int(params['mf_per_feature'])
    if 'n_rules' in params:
        out['n_rules'] = int(params['n_rules'])
    if 'mfs_per_input' in params:
        out['mfs_per_input'] = int(params['mfs_per_input'])
    if 'branch_rules' in params:
        out['branch_rules'] = int(params['branch_rules'])
    if 'top_rules' in params:
        out['top_rules'] = int(params['top_rules'])
    return out


def apply_saved_scaler_to_array(X_arr: np.ndarray, payload: dict[str, Any]) -> np.ndarray:
    mean = payload.get('scaler_mean')
    scale = payload.get('scaler_scale')
    X_arr = np.asarray(X_arr, dtype=np.float32)
    if mean is None or scale is None:
        return X_arr.astype(np.float32)

    mean_arr = np.asarray(mean, dtype=np.float32)
    scale_arr = np.asarray(scale, dtype=np.float32)
    safe_scale = np.where(scale_arr == 0, 1.0, scale_arr)
    return ((X_arr - mean_arr) / safe_scale).astype(np.float32)


def feature_names_from_payload(payload: dict[str, Any], fallback_columns: list[str]) -> list[str]:
    feature_names = payload.get('feature_names')
    if feature_names is None:
        n_features = int(payload.get('n_features', len(fallback_columns)))
        return list(fallback_columns[:n_features])
    return list(feature_names)


def preprocess_with_payload(X_processed_df: pd.DataFrame, payload: dict[str, Any]) -> tuple[pd.DataFrame, np.ndarray]:
    feature_names = feature_names_from_payload(payload, list(X_processed_df.columns))
    missing = [col for col in feature_names if col not in X_processed_df.columns]
    if missing:
        raise KeyError(f'Missing model-input columns for artifact: {missing[:10]}')
    X_sel = X_processed_df.loc[:, feature_names].copy()
    X_scaled = apply_saved_scaler_to_array(X_sel.values, payload)
    return X_sel, X_scaled


def load_cv_artifact(
    model_name: str,
    fold_idx: int,
    artifact_dir: Path | None = None,
    device: torch.device = DEVICE,
):
    artifact_dir = Path(artifact_dir or ARTIFACT_DATASET_DIR)
    fold_dir = artifact_dir / f'fold_{int(fold_idx):02d}'
    stem = MODEL_FILE_STEM[model_name]
    payload = torch.load(fold_dir / f'{stem}.pt', map_location=device)
    if not isinstance(payload, dict):
        raise TypeError(f'Unsupported checkpoint payload type: {type(payload)}')

    raw_params = payload.get('hparams') or payload.get('params') or {}
    state_dict = payload.get('model_state_dict') or payload.get('state_dict')
    if state_dict is None:
        raise KeyError('Checkpoint is missing model_state_dict/state_dict.')

    if model_name == 'GH-ANFIS':
        params = normalize_gh_params(raw_params)
        cfg = _infer_model_config(payload, params)
        model = GH_ANFIS(
            n_features=int(cfg['n_features']),
            n_outputs=int(cfg['n_outputs']),
            residual_rules=int(cfg.get('residual_rules', params.get('residual_rules', 8))),
            base_rules=int(cfg.get('base_rules', params.get('base_rules', 4))),
            mf_per_feature=int(cfg.get('mf_per_feature', params.get('mf_per_feature', 2))),
            device=device,
            residual_gate_mode=str(params.get('residual_gate_mode', 'complement')),
            rule_init_mode=str(params.get('rule_init_mode', 'balanced')),
            rule_seed=int(params.get('rule_seed', 0)),
            firing_mode=str(params.get('firing_mode', 'htsk')),
            use_input_norm=bool(params.get('use_input_norm', False)),
            enable_residual_branch=bool(params.get('enable_residual_branch', True)),
        ).to(device)
        model.load_state_dict(state_dict)

        if params.get('base_mask_threshold') is not None:
            model.base_mask_threshold = float(params['base_mask_threshold'])
        if params.get('residual_mask_threshold') is not None:
            model.residual_mask_threshold = float(params['residual_mask_threshold'])

        model.base_mask_frozen = bool(
            payload.get('base_mask_frozen', False)
            or params.get('base_hard_epochs', 0)
            or params.get('random_role_assignment', False)
        )
        model.residual_mask_frozen = bool(
            payload.get('residual_mask_frozen', False)
            or params.get('residual_hard_epochs', 0)
            or params.get('random_role_assignment', False)
        )
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        params = dict(raw_params)
        cfg = _infer_model_config(payload, params)
        n_inputs = int(cfg.get('n_inputs', cfg.get('n_features', payload['n_features'])))
        n_outputs = int(cfg.get('n_outputs', payload['n_outputs']))
        fusion_default = 'avg' if model_name == 'PH-ANFIS(Avg)' else 'stacked'
        fusion = str(cfg.get('fusion', params.get('fusion', fusion_default)))

        extra_meta = payload.get('extra_meta') or {}
        group_a_idx = list(cfg.get('group_a_idx', extra_meta.get('group_a_idx') or []))
        group_b_idx = list(cfg.get('group_b_idx', extra_meta.get('group_b_idx') or []))
        split_seed = int(cfg.get('rule_seed', params.get('split_seed', SEED)))

        if not group_a_idx or not group_b_idx:
            rng = np.random.default_rng(split_seed)
            order = np.arange(int(n_inputs), dtype=int)
            rng.shuffle(order)
            cut = int(max(1, n_inputs // 2))
            if cut >= n_inputs:
                cut = n_inputs - 1
            group_a_idx = np.sort(order[:cut]).tolist()
            group_b_idx = np.sort(order[cut:]).tolist()

        branch_rules = int(cfg.get('branch_rules', params.get('branch_rules', params.get('n_rules', 12))))
        top_rules = int(cfg.get('top_rules', params.get('top_rules', max(2, branch_rules // 2))))
        mfs_per_input = int(cfg.get('mfs_per_input', params.get('mfs_per_input', 3)))
        rule_init_mode = str(cfg.get('rule_init_mode', params.get('rule_init_mode', 'legacy')))
        rule_seed = int(cfg.get('rule_seed', params.get('rule_seed', split_seed)))
        firing_mode = str(cfg.get('firing_mode', params.get('firing_mode', 'htsk')))

        model = ParallelHierarchicalTSKANFIS(
            n_inputs=n_inputs,
            n_outputs=n_outputs,
            group_a_idx=group_a_idx,
            group_b_idx=group_b_idx,
            branch_rules=branch_rules,
            top_rules=top_rules,
            fusion=fusion,
            mfs_per_input=mfs_per_input,
            rule_init_mode=rule_init_mode,
            rule_seed=rule_seed,
            firing_mode=firing_mode,
        ).to(device)
        model.load_state_dict(state_dict)
    else:
        params = dict(raw_params)
        cfg = _infer_model_config(payload, params)
        n_inputs = int(cfg.get('n_inputs', cfg.get('n_features', payload['n_features'])))
        n_outputs = int(cfg.get('n_outputs', payload['n_outputs']))
        n_rules = int(cfg.get('n_rules', params.get('n_rules', 30)))
        mfs_per_input = int(cfg.get('mfs_per_input', params.get('mfs_per_input', 3)))
        rule_init_mode = str(cfg.get('rule_init_mode', params.get('rule_init_mode', 'legacy')))
        rule_seed = int(cfg.get('rule_seed', params.get('rule_seed', 0)))
        firing_mode = str(cfg.get('firing_mode', params.get('firing_mode', 'htsk')))

        model = TSKANFIS(
            n_inputs=n_inputs,
            n_rules=n_rules,
            n_outputs=n_outputs,
            mfs_per_input=mfs_per_input,
            rule_init_mode=rule_init_mode,
            rule_seed=rule_seed,
            firing_mode=firing_mode,
        ).to(device)
        model.load_state_dict(state_dict)

    model.eval()
    return model, payload


RAW_X_DF, PROCESSED_X_DF, Y_RAW, Y_ENC, RAW_CLASSES, RAW_TO_ENC = load_bcwd_views()
TRAIN_IDX, VAL_IDX = get_bcwd_fold_indices(Y_ENC, FOLD_IDX)
MALIGNANT_RAW_LABEL = int(RAW_CLASSES[-1])
BENIGN_RAW_LABEL = int(RAW_CLASSES[0])
MALIGNANT_ENC_LABEL = int(RAW_TO_ENC[MALIGNANT_RAW_LABEL])
ARTIFACT_DATASET_DIR = resolve_artifact_dataset_dir(
    DATASET_NAME,
    FOLD_IDX,
    MODEL_NAMES,
    preferred_dirname=ARTIFACT_DATASET_DIRNAME,
)

print('RAW_X_DF shape =', RAW_X_DF.shape)
print('PROCESSED_X_DF shape =', PROCESSED_X_DF.shape)
print('Raw classes =', RAW_CLASSES)
print('Malignant raw label =', MALIGNANT_RAW_LABEL)
print('Malignant encoded label =', MALIGNANT_ENC_LABEL)
print('Validation fold size =', len(VAL_IDX))
print('ARTIFACT_DATASET_DIR =', ARTIFACT_DATASET_DIR)


RAW_X_DF shape = (699, 9)
PROCESSED_X_DF shape = (699, 9)
Raw classes = [2, 4]
Malignant raw label = 4
Malignant encoded label = 1
Validation fold size = 140
ARTIFACT_DATASET_DIR = /home/harp3133t/Research/03_Research/GH-ANFIS_E404/hyper_parameter/cv_weights/Breast_Cancer_Wisconsin__Original___no_mi


In [4]:
def infer_term_label(centers_1d: np.ndarray, mf_idx: int) -> tuple[str, int]:
    low_idx = int(np.argmin(centers_1d))
    high_idx = int(np.argmax(centers_1d))
    if int(mf_idx) == low_idx:
        return 'low', -1
    if int(mf_idx) == high_idx:
        return 'high', 1
    return f'mid(mf{int(mf_idx) + 1})', 0


def humanize_feature_name(feature_name: Any) -> str:
    text = str(feature_name)
    if '=' in text:
        left, right = text.split('=', 1)
        return f'{left} == {right}'
    return text


def make_term_text(feature_name: Any, term_label: str) -> str:
    readable = humanize_feature_name(feature_name)
    if '=' in str(feature_name):
        if term_label == 'high':
            return f'{readable} is active'
        if term_label == 'low':
            return f'{readable} is inactive'
        return f'{readable} is partially active'
    return f'{readable} is {term_label}'


def tsk_rule_indices(model: TSKANFIS) -> np.ndarray:
    if hasattr(model, 'rule_mf_indices') and model.rule_mf_indices is not None:
        return model.rule_mf_indices.detach().cpu().numpy().astype(int)
    selector_logits = model.rule_mf_selector_logits.detach().cpu().numpy()
    return selector_logits.argmax(axis=-1).astype(int)


def build_tsk_rule_texts(model: TSKANFIS, feature_names: list[str]) -> list[str]:
    centers = model.centers.detach().cpu().numpy()
    rule_idx = tsk_rule_indices(model)
    texts = []
    for r in range(int(model.n_rules)):
        terms = []
        for d, feat in enumerate(feature_names):
            term_label, _ = infer_term_label(centers[d], int(rule_idx[r, d]))
            terms.append(make_term_text(feat, term_label))
        texts.append(' AND '.join(terms))
    return texts


@torch.no_grad()
def predict_binary_logits(model, x_tensor: torch.Tensor, batch_size: int = 512) -> np.ndarray:
    model.eval()
    outs = []
    n = int(x_tensor.shape[0])
    for start in range(0, n, batch_size):
        logits = model(x_tensor[start:start + batch_size])
        if logits.dim() == 2 and logits.size(1) == 1:
            logits = logits.squeeze(1)
        elif logits.dim() != 1:
            raise ValueError(f'Only binary single-logit models are supported here, got {tuple(logits.shape)}')
        outs.append(logits.detach().cpu().numpy())
    return np.concatenate(outs, axis=0)


@torch.no_grad()
def analyze_tsk_sample(
    model: TSKANFIS,
    x_tensor_2d: torch.Tensor,
    feature_names: list[str],
    top_k: int = 3,
    label: str = 'TSK-ANFIS',
):
    model.eval()
    logits, acts = model(x_tensor_2d, return_activations=True)
    if logits.dim() == 2 and logits.size(1) == 1:
        final_logit = float(logits.detach().cpu().reshape(-1)[0])
    elif logits.dim() == 1:
        final_logit = float(logits.detach().cpu().reshape(-1)[0])
    else:
        raise ValueError(f'Only binary single-logit models are supported here, got {tuple(logits.shape)}')

    final_prob = float(sigmoid_np(final_logit))
    pred_class = int(final_prob >= 0.5)

    x_arr = x_tensor_2d.detach().cpu().numpy()[0]
    rule_weights = acts['rule_weights'].detach().cpu().numpy()[0]
    rule_firing = acts.get('rule_firing')
    if rule_firing is not None:
        rule_firing = rule_firing.detach().cpu().numpy()[0]

    consequents = model.consequents.detach().cpu().numpy()
    rule_texts = build_tsk_rule_texts(model, feature_names)

    rows = []
    for r in range(int(model.n_rules)):
        rule_logit = float(np.dot(consequents[r, 0, :-1], x_arr) + consequents[r, 0, -1])
        rule_prob = float(sigmoid_np(rule_logit))
        rule_weight = float(rule_weights[r])
        firing = float(rule_firing[r]) if rule_firing is not None else np.nan
        rows.append(
            {
                'rule_idx': int(r),
                'rule_id': f'R{r + 1}',
                'rule_weight': rule_weight,
                'rule_firing': firing,
                'rule_logit': rule_logit,
                'rule_prob_class_1': rule_prob,
                'rule_contribution_logit': float(rule_weight * rule_logit),
                'antecedent_text': rule_texts[r],
                'if_then_text': f'IF {rule_texts[r]} THEN class_1 probability = {rule_prob:.4f} (rule_logit={rule_logit:.4f})',
            }
        )

    all_rules_df = (
        pd.DataFrame(rows)
        .sort_values(['rule_weight', 'rule_contribution_logit'], ascending=[False, False])
        .reset_index(drop=True)
    )
    return {
        'model_label': label,
        'final_logit': final_logit,
        'final_prob': final_prob,
        'pred_class': pred_class,
        'top_rules_df': all_rules_df.head(int(top_k)).copy(),
        'all_rules_df': all_rules_df,
    }


@torch.no_grad()
def analyze_ph_sample(
    model: ParallelHierarchicalTSKANFIS,
    x_tensor_2d: torch.Tensor,
    feature_names: list[str],
    top_k: int = 3,
    label: str = 'H-ANFIS',
):
    model.eval()
    y, acts = model(x_tensor_2d, return_activations=True)
    final_logit = float(y.detach().cpu().reshape(-1)[0])
    final_prob = float(sigmoid_np(final_logit))
    pred_class = int(final_prob >= 0.5)

    xa, xb = model.split_inputs(x_tensor_2d)
    feat_a = [feature_names[i] for i in model.group_a_idx]
    feat_b = [feature_names[i] for i in model.group_b_idx]

    branch_a_analysis = analyze_tsk_sample(model.branch_a, xa, feat_a, top_k=top_k, label='Branch A')
    branch_b_analysis = analyze_tsk_sample(model.branch_b, xb, feat_b, top_k=top_k, label='Branch B')

    out = {
        'model_label': label,
        'fusion': model.fusion,
        'final_logit': final_logit,
        'final_prob': final_prob,
        'pred_class': pred_class,
        'branch_a_analysis': branch_a_analysis,
        'branch_b_analysis': branch_b_analysis,
    }

    if model.fusion == 'avg':
        fusion_weight = float(torch.sigmoid(model.fusion_logits).detach().cpu().reshape(-1)[0])
        out['fusion_text'] = (
            f'final_logit = {fusion_weight:.4f} * branch_a_logit '
            f'+ {1.0 - fusion_weight:.4f} * branch_b_logit'
        )
        out['fusion_weight'] = fusion_weight
    else:
        top_input = acts['top_input']
        top_feature_names = [f'branch_output_{idx}' for idx in range(top_input.shape[1])]
        top_analysis = analyze_tsk_sample(model.top_anfis, top_input, top_feature_names, top_k=top_k, label='Top fusion ANFIS')
        out['fusion_text'] = 'stacked top-level ANFIS combines the two branch outputs.'
        out['top_analysis'] = top_analysis

    return out


def gh_branch_masks(model: GH_ANFIS) -> dict[str, np.ndarray]:
    base_soft = torch.sigmoid(model.base_mask_logits).detach().cpu().numpy()
    base_hard = (base_soft >= float(model.base_mask_threshold)).astype(float)

    residual_soft = torch.sigmoid(model.residual_mask_logits).detach().cpu().numpy()
    residual_hard = (residual_soft >= float(model.residual_mask_threshold)).astype(float)

    if bool(model.residual_use_complement):
        residual_effective_hard = residual_hard * (1.0 - base_hard)
    else:
        residual_effective_hard = residual_hard.copy()

    return {
        'base_soft': base_soft,
        'base_hard': base_hard,
        'residual_soft': residual_soft,
        'residual_hard': residual_hard,
        'residual_effective_hard': residual_effective_hard,
    }


def gh_module_pack(model: GH_ANFIS, module: str) -> dict[str, Any]:
    masks = gh_branch_masks(model)
    if module == 'base':
        return {
            'module': 'base',
            'module_label': 'Primary',
            'display_prefix': 'P',
            'internal_prefix': 'B',
            'centers': model.s_mf_centers.detach().cpu().numpy(),
            'selector_logits': model.s_rule_mf_selector_logits.detach().cpu().numpy(),
            'consequents': model.base_consequents,
            'gate_hard': masks['base_hard'],
            'n_rules': int(model.base_rules),
        }
    if module == 'residual':
        return {
            'module': 'residual',
            'module_label': 'Complementary',
            'display_prefix': 'C',
            'internal_prefix': 'R',
            'centers': model.p_mf_centers.detach().cpu().numpy(),
            'selector_logits': model.p_rule_mf_selector_logits.detach().cpu().numpy(),
            'consequents': model.residual_consequents,
            'gate_hard': masks['residual_effective_hard'],
            'n_rules': int(model.residual_rules),
        }
    raise ValueError("module must be 'base' or 'residual'")


@torch.no_grad()
def gh_predict_binary_logits(
    model: GH_ANFIS,
    x_tensor: torch.Tensor,
    phase: str,
    mode: str,
    batch_size: int = 512,
) -> np.ndarray:
    model.eval()
    model.set_phase(phase)
    model.set_mode(mode)
    outs = []
    n = int(x_tensor.shape[0])
    for start in range(0, n, batch_size):
        logits = model(x_tensor[start:start + batch_size])
        if logits.dim() == 2 and logits.size(1) == 1:
            logits = logits.squeeze(1)
        elif logits.dim() != 1:
            raise ValueError(f'Only binary single-logit GH models are supported here, got {tuple(logits.shape)}')
        outs.append(logits.detach().cpu().numpy())
    return np.concatenate(outs, axis=0)


def gh_module_rule_details(
    model: GH_ANFIS,
    x_tensor_2d: torch.Tensor,
    feature_names: list[str],
    module: str,
    acts: dict[str, Any],
    top_terms: int = 8,
) -> pd.DataFrame:
    mp = gh_module_pack(model, module)

    selector_logits = np.asarray(mp['selector_logits'], dtype=np.float64)
    selector_logits = selector_logits - selector_logits.max(axis=-1, keepdims=True)
    selector_probs = np.exp(selector_logits)
    selector_probs = selector_probs / selector_probs.sum(axis=-1, keepdims=True)

    centers = np.asarray(mp['centers'], dtype=np.float64)
    x_arr = x_tensor_2d.detach().cpu().numpy()[0]

    if module == 'base':
        gate_vec = acts['gate_base'][0].detach().cpu().numpy()
        rule_weights = acts['base_rule_weights'][0].detach().cpu().numpy()
    else:
        gate_vec = acts['gate_residual'][0].detach().cpu().numpy()
        rule_weights = acts['residual_rule_weights'][0].detach().cpu().numpy()

    rows = []
    for r in range(mp['n_rules']):
        best_mf = selector_probs[r].argmax(axis=1)
        best_prob = selector_probs[r].max(axis=1)

        active_idx = np.where(gate_vec > 1e-8)[0].tolist()
        if active_idx:
            active_idx = sorted(active_idx, key=lambda d: float(best_prob[d]), reverse=True)
        else:
            active_idx = np.argsort(best_prob)[::-1].tolist()
        active_idx = active_idx[: max(1, int(top_terms))]

        terms = []
        for d in active_idx:
            term_label, _ = infer_term_label(centers[d], int(best_mf[d]))
            terms.append(make_term_text(feature_names[d], term_label))
        antecedent_text = ' AND '.join(terms) if terms else '(no active terms)'

        layer = mp['consequents'][r]
        weight = layer.weight.detach().cpu().numpy().reshape(-1)
        bias = float(layer.bias.detach().cpu().numpy().reshape(-1)[0]) if layer.bias is not None else 0.0
        feat_weight = weight[: len(feature_names)]
        bias_weight = float(weight[len(feature_names)]) if len(weight) > len(feature_names) else 0.0
        rule_logit = float(np.sum(x_arr * feat_weight * gate_vec) + bias_weight + bias)
        rule_prob = float(sigmoid_np(rule_logit))
        rule_weight = float(rule_weights[r])

        rows.append(
            {
                'module': module,
                'module_label': mp['module_label'],
                'rule_idx': int(r),
                'display_rule_id': f"{mp['display_prefix']}{r + 1}",
                'internal_rule_id': f"{mp['internal_prefix']}{r + 1}",
                'rule_weight': rule_weight,
                'rule_logit': rule_logit,
                'rule_prob_class_1': rule_prob,
                'rule_contribution_logit': float(rule_weight * rule_logit),
                'antecedent_text': antecedent_text,
                'if_then_text': f'IF {antecedent_text} THEN class_1 probability = {rule_prob:.4f} (rule_logit={rule_logit:.4f})',
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(['rule_weight', 'rule_contribution_logit'], ascending=[False, False])
        .reset_index(drop=True)
    )


@torch.no_grad()
def analyze_gh_sample(
    model: GH_ANFIS,
    x_tensor_2d: torch.Tensor,
    feature_names: list[str],
    top_k_rules: int = 3,
    top_terms: int = 8,
    label: str = 'GRS-ANFIS',
):
    base_logit = float(gh_predict_binary_logits(model, x_tensor_2d, phase='base', mode='base_only', batch_size=1)[0])
    residual_logit = float(gh_predict_binary_logits(model, x_tensor_2d, phase='residual_complement', mode='residual_only', batch_size=1)[0])
    full_logit = float(gh_predict_binary_logits(model, x_tensor_2d, phase='residual_complement', mode='full', batch_size=1)[0])

    base_prob = float(sigmoid_np(base_logit))
    residual_prob = float(sigmoid_np(residual_logit))
    full_prob = float(sigmoid_np(full_logit))

    model.eval()
    model.set_phase('residual_complement')
    model.set_mode('full')
    _, acts = model(x_tensor_2d, return_activations=True, use_soft_eval=False)

    base_rules_df = gh_module_rule_details(model, x_tensor_2d, feature_names, module='base', acts=acts, top_terms=top_terms)
    residual_rules_df = gh_module_rule_details(model, x_tensor_2d, feature_names, module='residual', acts=acts, top_terms=top_terms)

    return {
        'model_label': label,
        'base_logit': base_logit,
        'base_prob': base_prob,
        'base_pred': int(base_prob >= 0.5),
        'residual_logit': residual_logit,
        'residual_prob': residual_prob,
        'residual_pred': int(residual_prob >= 0.5),
        'full_logit': full_logit,
        'full_prob': full_prob,
        'full_pred': int(full_prob >= 0.5),
        'delta_full_minus_base': float(full_prob - base_prob),
        'delta_residual_logit': float(full_logit - base_logit),
        'base_rules_df': base_rules_df.head(int(top_k_rules)).copy(),
        'residual_rules_df': residual_rules_df.head(int(top_k_rules)).copy(),
        'all_base_rules_df': base_rules_df,
        'all_residual_rules_df': residual_rules_df,
    }


def find_interesting_gh_case(
    model: GH_ANFIS,
    payload: dict[str, Any],
    X_processed_df: pd.DataFrame,
    y_raw: np.ndarray,
    y_encoded: np.ndarray,
    val_idx: np.ndarray,
    min_base_positive_prob: float = 0.8,
):
    X_val_processed = X_processed_df.iloc[val_idx].copy()
    _, X_val_scaled = preprocess_with_payload(X_val_processed, payload)
    x_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=DEVICE)

    base_logits = gh_predict_binary_logits(model, x_tensor, phase='base', mode='base_only')
    residual_logits = gh_predict_binary_logits(model, x_tensor, phase='residual_complement', mode='residual_only')
    full_logits = gh_predict_binary_logits(model, x_tensor, phase='residual_complement', mode='full')

    df = pd.DataFrame(
        {
            'orig_index': val_idx,
            'y_raw': y_raw[val_idx],
            'y_encoded': y_encoded[val_idx],
            'base_prob': sigmoid_np(base_logits),
            'residual_prob': sigmoid_np(residual_logits),
            'full_prob': sigmoid_np(full_logits),
            'base_logit': base_logits,
            'residual_logit': residual_logits,
            'full_logit': full_logits,
        }
    )
    df['base_pred'] = (df['base_prob'] >= 0.5).astype(int)
    df['residual_pred'] = (df['residual_prob'] >= 0.5).astype(int)
    df['full_pred'] = (df['full_prob'] >= 0.5).astype(int)
    df['delta_full_minus_base'] = df['full_prob'] - df['base_prob']
    df['prediction_flip'] = df['base_pred'] != df['full_pred']

    positive_df = df[df['y_encoded'] == MALIGNANT_ENC_LABEL].copy()
    if positive_df.empty:
        raise ValueError('No malignant validation cases were found for this fold.')

    flip_df = positive_df[
        (positive_df['base_pred'] == MALIGNANT_ENC_LABEL)
        & (positive_df['full_pred'] != positive_df['base_pred'])
    ].copy()

    if not flip_df.empty:
        selected = flip_df.sort_values(['full_prob', 'delta_full_minus_base'], ascending=[True, True]).iloc[0]
        reason = 'base는 malignant로 보지만 full에서는 예측이 뒤집히는 malignant 케이스'
    else:
        strong_df = positive_df[positive_df['base_prob'] >= float(min_base_positive_prob)].copy()
        if strong_df.empty:
            strong_df = positive_df.copy()
        selected = strong_df.sort_values(['delta_full_minus_base', 'full_prob'], ascending=[True, True]).iloc[0]
        reason = 'base 대비 full malignant 확률 하락폭이 가장 큰 malignant 케이스'

    candidate_table = positive_df.sort_values(
        ['prediction_flip', 'delta_full_minus_base', 'full_prob'],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    return candidate_table, int(selected['orig_index']), reason


def describe_gh_case(analysis: dict[str, Any]) -> str:
    base_prob = float(analysis['base_prob'])
    residual_prob = float(analysis['residual_prob'])
    full_prob = float(analysis['full_prob'])
    delta = float(analysis['delta_full_minus_base'])

    if analysis['base_pred'] != analysis['full_pred']:
        direction = (
            f'Primary(base)는 class_{analysis["base_pred"]}로 보지만 '
            f'Complementary를 합친 full은 class_{analysis["full_pred"]}로 뒤집힙니다.'
        )
    elif delta < 0:
        direction = f'Complementary를 합치면 class_1 probability가 {base_prob:.4f} -> {full_prob:.4f}로 낮아집니다.'
    elif delta > 0:
        direction = f'Complementary를 합치면 class_1 probability가 {base_prob:.4f} -> {full_prob:.4f}로 높아집니다.'
    else:
        direction = f'Complementary를 합쳐도 class_1 probability는 {full_prob:.4f}로 거의 변하지 않습니다.'

    base_rule = analysis['base_rules_df'].iloc[0]['if_then_text'] if not analysis['base_rules_df'].empty else '(no primary rule)'
    residual_rule = analysis['residual_rules_df'].iloc[0]['if_then_text'] if not analysis['residual_rules_df'].empty else '(no complementary rule)'

    return (
        f'{direction} '
        f'base_prob={base_prob:.4f}, residual_prob={residual_prob:.4f}, full_prob={full_prob:.4f}. '
        f'Top Primary rule: {base_rule} '
        f'Top Complementary rule: {residual_rule}'
    )


def build_model_input_view(sample_processed_df: pd.DataFrame) -> pd.DataFrame:
    row = sample_processed_df.iloc[0]
    active = row[row != 0].sort_values(ascending=False).reset_index()
    active.columns = ['feature', 'value']
    active['feature_readable'] = active['feature'].map(humanize_feature_name)
    return active[['feature_readable', 'feature', 'value']]


In [5]:
gh_model, gh_payload = load_cv_artifact('GH-ANFIS', FOLD_IDX)
case_candidates_df, selected_case_index, case_selection_reason = find_interesting_gh_case(
    gh_model,
    gh_payload,
    PROCESSED_X_DF,
    Y_RAW,
    Y_ENC,
    VAL_IDX,
    min_base_positive_prob=MIN_BASE_POSITIVE_PROB,
)

if CASE_INDEX_OVERRIDE is not None:
    selected_case_index = int(CASE_INDEX_OVERRIDE)
    case_selection_reason = 'manual override'

selected_case_raw = RAW_X_DF.iloc[[selected_case_index]].copy()
selected_case_processed = PROCESSED_X_DF.iloc[[selected_case_index]].copy()
selected_case_input_view = build_model_input_view(selected_case_processed)
selected_case_target_raw = int(Y_RAW[selected_case_index])
selected_case_target_encoded = int(Y_ENC[selected_case_index])
selected_case_is_val = bool(int(selected_case_index) in set(VAL_IDX.tolist()))

print('Selected case index:', selected_case_index)
print('Selection reason:', case_selection_reason)
print('Raw target:', selected_case_target_raw)
print('Encoded target:', selected_case_target_encoded)
print('Is in validation fold:', selected_case_is_val)
print('Malignant raw label:', MALIGNANT_RAW_LABEL)
print('Benign raw label:', BENIGN_RAW_LABEL)
print('Model input feature count:', len(selected_case_input_view))

print('Top candidate cases (malignant validation cases only):')
display(case_candidates_df.head(20))

print('Selected patient raw feature values (9 original variables):')
display(selected_case_raw.T.rename(columns={selected_case_index: 'value'}))

print('Selected patient model input feature values (before scaling):')
display(selected_case_input_view.head(9))


Selected case index: 231
Selection reason: base 대비 full malignant 확률 하락폭이 가장 큰 malignant 케이스
Raw target: 4
Encoded target: 1
Is in validation fold: True
Malignant raw label: 4
Benign raw label: 2
Model input feature count: 9
Top candidate cases (malignant validation cases only):


,orig_index,y_raw,y_encoded,base_prob,residual_prob,full_prob,base_logit,residual_logit,full_logit,base_pred,residual_pred,full_pred,delta_full_minus_base,prediction_flip
0,489,4,1,0.469754,0.620872,0.591972,-0.121130,0.493253,0.372123,0,1,1,0.122217,True
1,488,4,1,0.182161,0.987417,0.945881,-1.501776,4.362712,2.860936,0,1,1,0.763720,True
2,231,4,1,0.955629,0.135628,0.771659,3.069787,-1.852084,1.217703,1,0,1,-0.183970,False
3,166,4,1,0.951609,0.185369,0.817344,2.978844,-1.480386,1.498458,1,0,1,-0.134265,False
4,449,4,1,0.986054,0.203848,0.947654,4.258527,-1.362415,2.896112,1,0,1,-0.038400,False
5,453,4,1,0.991940,0.147843,0.955262,4.812788,-1.751620,3.061168,1,0,1,-0.036678,False
6,218,4,1,0.940810,0.416427,0.918977,2.765990,-0.337459,2.428532,1,0,1,-0.021833,False
7,74,4,1,0.986693,0.360443,0.976630,4.306084,-0.573442,3.732642,1,0,1,-0.010063,False
8,54,4,1,0.998537,0.114483,0.988792,6.525644,-2.045748,4.479896,1,0,1,-0.009744,False
9,98,4,1,0.999558,0.067101,0.993896,7.724715,-2.632098,5.092617,1,0,1,-0.005663,False


Selected patient raw feature values (9 original variables):


,value
Clump_thickness,6.0
Uniformity_of_cell_size,8.0
Uniformity_of_cell_shape,7.0
Marginal_adhesion,5.0
Single_epithelial_cell_size,6.0
Bare_nuclei,8.0
Bland_chromatin,8.0
Normal_nucleoli,9.0
Mitoses,2.0


Selected patient model input feature values (before scaling):


,feature_readable,feature,value
0,Normal_nucleoli,Normal_nucleoli,9.0
1,Bare_nuclei,Bare_nuclei,8.0
2,Uniformity_of_cell_size,Uniformity_of_cell_size,8.0
3,Bland_chromatin,Bland_chromatin,8.0
4,Uniformity_of_cell_shape,Uniformity_of_cell_shape,7.0
5,Clump_thickness,Clump_thickness,6.0
6,Single_epithelial_cell_size,Single_epithelial_cell_size,6.0
7,Marginal_adhesion,Marginal_adhesion,5.0
8,Mitoses,Mitoses,2.0


In [6]:
analysis_by_model = {}
summary_rows = []

for model_name in MODEL_NAMES:
    model, payload = load_cv_artifact(model_name, FOLD_IDX)
    sample_feature_df, sample_scaled = preprocess_with_payload(selected_case_processed, payload)
    sample_tensor = torch.tensor(sample_scaled, dtype=torch.float32, device=DEVICE)

    if model_name == 'GH-ANFIS':
        analysis = analyze_gh_sample(
            model,
            sample_tensor,
            list(sample_feature_df.columns),
            top_k_rules=TOP_RULES_PER_MODEL,
            top_terms=TOP_TERMS_PER_RULE,
            label=MODEL_DISPLAY_NAMES[model_name],
        )
        summary_rows.append(
            {
                'artifact_name': model_name,
                'model': MODEL_DISPLAY_NAMES[model_name],
                'n_features_used': int(sample_feature_df.shape[1]),
                'predicted_class': int(analysis['full_pred']),
                'class_1_probability': float(analysis['full_prob']),
                'base_probability': float(analysis['base_prob']),
                'residual_probability': float(analysis['residual_prob']),
                'delta_full_minus_base': float(analysis['delta_full_minus_base']),
                'top_rule_summary': (
                    f"base={analysis['base_rules_df'].iloc[0]['display_rule_id'] if not analysis['base_rules_df'].empty else 'NA'}; "
                    f"residual={analysis['residual_rules_df'].iloc[0]['display_rule_id'] if not analysis['residual_rules_df'].empty else 'NA'}"
                ),
            }
        )
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        analysis = analyze_ph_sample(
            model,
            sample_tensor,
            list(sample_feature_df.columns),
            top_k=TOP_RULES_PER_MODEL,
            label=MODEL_DISPLAY_NAMES[model_name],
        )
        top_rule_summary = 'branch_a=' + analysis['branch_a_analysis']['top_rules_df'].iloc[0]['rule_id']
        top_rule_summary += '; branch_b=' + analysis['branch_b_analysis']['top_rules_df'].iloc[0]['rule_id']
        if 'top_analysis' in analysis:
            top_rule_summary += '; top=' + analysis['top_analysis']['top_rules_df'].iloc[0]['rule_id']

        summary_rows.append(
            {
                'artifact_name': model_name,
                'model': MODEL_DISPLAY_NAMES[model_name],
                'n_features_used': int(sample_feature_df.shape[1]),
                'predicted_class': int(analysis['pred_class']),
                'class_1_probability': float(analysis['final_prob']),
                'base_probability': np.nan,
                'residual_probability': np.nan,
                'delta_full_minus_base': np.nan,
                'top_rule_summary': top_rule_summary,
            }
        )
    else:
        analysis = analyze_tsk_sample(
            model,
            sample_tensor,
            list(sample_feature_df.columns),
            top_k=TOP_RULES_PER_MODEL,
            label=MODEL_DISPLAY_NAMES[model_name],
        )
        summary_rows.append(
            {
                'artifact_name': model_name,
                'model': MODEL_DISPLAY_NAMES[model_name],
                'n_features_used': int(sample_feature_df.shape[1]),
                'predicted_class': int(analysis['pred_class']),
                'class_1_probability': float(analysis['final_prob']),
                'base_probability': np.nan,
                'residual_probability': np.nan,
                'delta_full_minus_base': np.nan,
                'top_rule_summary': analysis['top_rules_df'].iloc[0]['rule_id'],
            }
        )

    analysis['payload'] = payload
    analysis['sample_feature_df'] = sample_feature_df
    analysis_by_model[model_name] = analysis

comparison_df = pd.DataFrame(summary_rows)
comparison_df['true_encoded_class'] = selected_case_target_encoded
comparison_df['true_raw_target'] = selected_case_target_raw
comparison_df['correct'] = comparison_df['predicted_class'] == selected_case_target_encoded
comparison_df['sort_key'] = comparison_df['artifact_name'].map({name: idx for idx, name in enumerate(MODEL_NAMES)})
comparison_df = comparison_df.sort_values('sort_key').drop(columns=['sort_key']).reset_index(drop=True)

print('Model comparison summary for the selected malignant case:')
display(comparison_df)


Model comparison summary for the selected malignant case:


,artifact_name,model,n_features_used,predicted_class,class_1_probability,base_probability,residual_probability,delta_full_minus_base,top_rule_summary,true_encoded_class,true_raw_target,correct
0,ANFIS,ANFIS,9,1,0.806997,NaN,NaN,NaN,R28,1,4,True
1,GA-ANFIS,GA-ANFIS,5,1,0.996833,NaN,NaN,NaN,R15,1,4,True
2,PSO-ANFIS,PSO-ANFIS,7,1,0.999661,NaN,NaN,NaN,R16,1,4,True
3,PH-ANFIS(Avg),H-ANFIS,9,1,0.986705,NaN,NaN,NaN,branch_a=R16; branch_b=R15,1,4,True
4,GH-ANFIS,GRS-ANFIS,9,1,0.771659,0.955629,0.135628,-0.18397,base=P4; residual=C2,1,4,True


In [7]:
gh_analysis = analysis_by_model['GH-ANFIS']
gh_narrative = describe_gh_case(gh_analysis)

print('GRS-focused interpretation:')
print(gh_narrative)

for model_name in MODEL_NAMES:
    analysis = analysis_by_model[model_name]
    display_name = MODEL_DISPLAY_NAMES[model_name]

    print('=' * 100)
    print(f'[{display_name}] artifact={model_name}')

    if model_name == 'GH-ANFIS':
        print(f"base_prob={analysis['base_prob']:.4f}, residual_prob={analysis['residual_prob']:.4f}, full_prob={analysis['full_prob']:.4f}")
        print(f"delta_full_minus_base={analysis['delta_full_minus_base']:.4f}, full_pred=class_{analysis['full_pred']}")
        print('[Top Primary/base rules]')
        display(analysis['base_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        print('[Top Complementary/residual rules]')
        display(analysis['residual_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        print(f"final_prob={analysis['final_prob']:.4f}, pred=class_{analysis['pred_class']}")
        print('fusion:', analysis['fusion_text'])
        print('[Branch A top rules]')
        display(analysis['branch_a_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        print('[Branch B top rules]')
        display(analysis['branch_b_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        if 'top_analysis' in analysis:
            print('[Top fusion ANFIS rules]')
            display(analysis['top_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
    else:
        print(f"final_prob={analysis['final_prob']:.4f}, pred=class_{analysis['pred_class']}")
        display(analysis['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])


GRS-focused interpretation:
Complementary를 합치면 class_1 probability가 0.9556 -> 0.7717로 낮아집니다. base_prob=0.9556, residual_prob=0.1356, full_prob=0.7717. Top Primary rule: IF Clump_thickness is high AND Uniformity_of_cell_shape is high AND Bare_nuclei is high AND Bland_chromatin is high AND Normal_nucleoli is high AND Mitoses is high THEN class_1 probability = 0.8814 (rule_logit=2.0058) Top Complementary rule: IF Uniformity_of_cell_size is high AND Marginal_adhesion is high AND Single_epithelial_cell_size is low THEN class_1 probability = 0.0077 (rule_logit=-4.8604)
[ANFIS] artifact=ANFIS
final_prob=0.8070, pred=class_1


,rule_id,rule_weight,rule_prob_class_1,if_then_text
0,R28,0.453269,0.001666,IF Clump_thickness is mid(mf1) AND Uniformity_...
1,R29,0.106883,0.999991,IF Clump_thickness is low AND Uniformity_of_ce...
2,R30,0.100759,0.097419,IF Clump_thickness is high AND Uniformity_of_c...


[GA-ANFIS] artifact=GA-ANFIS
final_prob=0.9968, pred=class_1


,rule_id,rule_weight,rule_prob_class_1,if_then_text
0,R15,0.295378,0.997588,IF Clump_thickness is mid(mf3) AND Single_epit...
1,R12,0.271804,0.997636,IF Clump_thickness is mid(mf3) AND Single_epit...
2,R18,0.209767,0.997136,IF Clump_thickness is mid(mf3) AND Single_epit...


[PSO-ANFIS] artifact=PSO-ANFIS
final_prob=0.9997, pred=class_1


...

,rule_id,rule_weight,rule_prob_class_1,if_then_text
0,R15,0.529015,0.982406,IF Clump_thickness is low AND Uniformity_of_ce...
1,R7,0.207192,0.979747,IF Clump_thickness is low AND Uniformity_of_ce...
2,R16,0.097142,0.991899,IF Clump_thickness is high AND Uniformity_of_c...


[GRS-ANFIS] artifact=GH-ANFIS
base_prob=0.9556, residual_prob=0.1356, full_prob=0.7717
delta_full_minus_base=-0.1840, full_pred=class_1
[Top Primary/base rules]


,display_rule_id,rule_weight,rule_prob_class_1,if_then_text
0,P4,0.811138,0.881409,IF Clump_thickness is high AND Uniformity_of_c...
1,P6,0.092732,0.999966,IF Clump_thickness is low AND Uniformity_of_ce...
2,P1,0.084164,0.991475,IF Clump_thickness is high AND Uniformity_of_c...


[Top Complementary/residual rules]


,display_rule_id,rule_weight,rule_prob_class_1,if_then_text
0,C2,0.213846,0.007688,IF Uniformity_of_cell_size is high AND Margina...
1,C9,0.135288,0.026354,IF Uniformity_of_cell_size is low AND Marginal...
2,C1,0.135288,0.026344,IF Uniformity_of_cell_size is low AND Marginal...


In [8]:
def df_to_records(df: pd.DataFrame) -> list[dict[str, Any]]:
    if df is None or df.empty:
        return []
    return json.loads(df.to_json(orient='records', force_ascii=False))


stem = f'bcwd_fold_{int(FOLD_IDX):02d}_case_{int(selected_case_index):03d}'
comparison_csv = EXPORT_DIR / f'{stem}_comparison.csv'
candidates_csv = EXPORT_DIR / f'{stem}_gh_candidates.csv'
selected_case_raw_csv = EXPORT_DIR / f'{stem}_raw_features.csv'
selected_case_input_view_csv = EXPORT_DIR / f'{stem}_model_input_features.csv'
top_rules_json = EXPORT_DIR / f'{stem}_top_rules.json'
summary_json = EXPORT_DIR / f'{stem}_summary.json'

comparison_df.to_csv(comparison_csv, index=False)
case_candidates_df.to_csv(candidates_csv, index=False)
selected_case_raw.T.rename(columns={selected_case_index: 'value'}).to_csv(selected_case_raw_csv)
selected_case_input_view.to_csv(selected_case_input_view_csv, index=False)

rules_payload = {}
for model_name in MODEL_NAMES:
    analysis = analysis_by_model[model_name]
    display_name = MODEL_DISPLAY_NAMES[model_name]
    if model_name == 'GH-ANFIS':
        rules_payload[display_name] = {
            'base_rules': df_to_records(analysis['base_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
            'residual_rules': df_to_records(analysis['residual_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
        }
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        payload = {
            'branch_a_rules': df_to_records(analysis['branch_a_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
            'branch_b_rules': df_to_records(analysis['branch_b_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
        }
        if 'top_analysis' in analysis:
            payload['top_fusion_rules'] = df_to_records(analysis['top_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        rules_payload[display_name] = payload
    else:
        rules_payload[display_name] = {
            'rules': df_to_records(analysis['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
        }

top_rules_json.write_text(json.dumps(rules_payload, ensure_ascii=False, indent=2), encoding='utf-8')

summary_payload = {
    'dataset_name': DATASET_NAME,
    'artifact_dataset_dir': str(ARTIFACT_DATASET_DIR),
    'fold_idx': int(FOLD_IDX),
    'selected_case_index': int(selected_case_index),
    'selection_reason': case_selection_reason,
    'selected_case_target_raw': int(selected_case_target_raw),
    'selected_case_target_encoded': int(selected_case_target_encoded),
    'is_validation_case': bool(selected_case_is_val),
    'label_mapping_raw_to_encoded': {str(k): int(v) for k, v in RAW_TO_ENC.items()},
    'gh_narrative': gh_narrative,
    'models': {
        row['artifact_name']: {
            'probability_class_1': float(row['class_1_probability']),
            'predicted_class': int(row['predicted_class']),
            'correct': bool(row['correct']),
        }
        for _, row in comparison_df.iterrows()
    },
}
summary_json.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('Exported:')
print('  comparison_csv =', comparison_csv.resolve())
print('  candidates_csv =', candidates_csv.resolve())
print('  selected_case_raw_csv =', selected_case_raw_csv.resolve())
print('  selected_case_input_view_csv =', selected_case_input_view_csv.resolve())
print('  top_rules_json =', top_rules_json.resolve())
print('  summary_json =', summary_json.resolve())


Exported:
  comparison_csv = /home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/bcwd_case_model_interpretation/bcwd_fold_04_case_231_comparison.csv
  candidates_csv = /home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/bcwd_case_model_interpretation/bcwd_fold_04_case_231_gh_candidates.csv
  selected_case_raw_csv = /home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/bcwd_case_model_interpretation/bcwd_fold_04_case_231_raw_features.csv
  selected_case_input_view_csv = /home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/bcwd_case_model_interpretation/bcwd_fold_04_case_231_model_input_features.csv
  top_rules_json = /home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/bcwd_case_model_interpretation/bcwd_fold_04_case_231_top_rules.json
  summary_json = /home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/bcwd_case_model_interpretation/bcwd_fold_04_case_231_summary.json
